# 文本相似度实例

## Step1 导入相关包

In [1]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset

## Step2 加载数据集

In [2]:
dataset = load_dataset("json", data_files="./train_pair_1w.json", split="train")
dataset

Dataset({
    features: ['sentence1', 'sentence2', 'label'],
    num_rows: 10000
})

In [3]:
dataset[0]

{'sentence1': '找一部小时候的动画片', 'sentence2': '求一部小时候的动画片。谢了', 'label': '1'}

## Step3 划分数据集

In [4]:
datasets = dataset.train_test_split(test_size=0.2)
datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label'],
        num_rows: 8000
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label'],
        num_rows: 2000
    })
})

## Step4 数据集预处理

In [5]:
import torch

tokenizer = AutoTokenizer.from_pretrained("../../model/hfl/chinese-macbert-base")

def process_function(examples):
    tokenized_examples = tokenizer(examples["sentence1"], examples["sentence2"], max_length=128, truncation=True)
    tokenized_examples["labels"] = [float(label) for label in examples["label"]]
    return tokenized_examples

tokenized_datasets = datasets.map(process_function, batched=True, remove_columns=datasets["train"].column_names)
tokenized_datasets

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 8000
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 2000
    })
})

In [6]:
tokenized_datasets["train"][0]

{'input_ids': [101,
  3219,
  1921,
  6820,
  3341,
  749,
  2145,
  782,
  511,
  102,
  1923,
  782,
  8024,
  7350,
  2209,
  6564,
  5031,
  6887,
  8024,
  1963,
  3362,
  2769,
  812,
  1086,
  679,
  6230,
  2533,
  800,
  4638,
  711,
  782,
  3300,
  6637,
  8024,
  2769,
  812,
  738,
  4802,
  2141,
  1922,
  7410,
  6374,
  1962,
  1568,
  8024,
  671,
  702,
  769,
  2518,
  1282,
  2399,
  4638,
  3301,
  1351,
  738,
  679,
  833,
  1963,
  800,
  6821,
  3416,
  2521,
  2769,
  812,
  3291,
  1962,
  4638,
  749,
  8024,
  800,
  2578,
  2428,
  7770,
  7414,
  8024,
  2418,
  802,
  2341,
  1975,
  8024,
  4851,
  6505,
  1453,
  1168,
  8024,
  3209,
  3227,
  3221,
  671,
  855,
  769,
  7354,
  1767,
  4638,
  782,
  4289,
  511,
  102],
 'token_type_ids': [0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,

## Step5 创建模型

In [7]:
from transformers import BertForSequenceClassification 
model = AutoModelForSequenceClassification.from_pretrained("../../model/hfl/chinese-macbert-base", num_labels=1)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at ../../model/hfl/chinese-macbert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## Step6 创建评估函数

In [8]:
import evaluate

acc_metric = evaluate.load("./metric_accuracy.py")
f1_metirc = evaluate.load("../../evaluate/metrics/f1") # 提供的metric_f1版本不兼容

In [ ]:

def eval_metric(eval_predict):
    predictions, labels = eval_predict
    
    # predictions = [int(p > 0.5) for p in predictions]
    # labels = [int(l) for l in labels]
    
    # 使用 numpy 进行批量比较和类型转换
    predictions = (predictions > 0.5).astype(int) 
    labels = labels.astype(int)
    
    # while num_label = 2
    # predictions = predictions.argmax(axis=-1)
    
    acc = acc_metric.compute(predictions=predictions, references=labels)
    f1 = f1_metirc.compute(predictions=predictions, references=labels)
    acc.update(f1)
    return acc

## Step7 创建TrainingArguments

In [ ]:
train_args = TrainingArguments(output_dir="./cross_model",      # 输出文件夹
                               per_device_train_batch_size=32,  # 训练时的batch_size
                               per_device_eval_batch_size=32,   # 验证时的batch_size
                               logging_steps=10,                # log 打印的频率
                               eval_strategy="epoch",           # 评估策略
                               save_strategy="epoch",           # 保存策略
                               save_total_limit=3,              # 最大保存数
                               learning_rate=2e-5,              # 学习率
                               weight_decay=0.01,               # weight_decay
                               metric_for_best_model="f1",      # 设定评估指标
                               load_best_model_at_end=True)     # 训练完成后加载最优模型
train_args

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
dispatch_batches=None,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=epoch,
evaluation_strategy=None,
fp16=False,
fp16_backend=auto,
fp

## Step8 创建Trainer

In [ ]:
from transformers import DataCollatorWithPadding
trainer = Trainer(model=model, 
                  args=train_args, 
                  tokenizer=tokenizer,
                  train_dataset=tokenized_datasets["train"], 
                  eval_dataset=tokenized_datasets["test"], 
                  data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
                  compute_metrics=eval_metric)

## Step9 模型训练

In [ ]:
trainer.train()

  0%|          | 0/750 [00:00<?, ?it/s]

{'loss': 0.3167, 'grad_norm': 7.415092468261719, 'learning_rate': 1.9733333333333336e-05, 'epoch': 0.04}
{'loss': 0.177, 'grad_norm': 15.541095733642578, 'learning_rate': 1.9466666666666668e-05, 'epoch': 0.08}
{'loss': 0.1658, 'grad_norm': 12.618912696838379, 'learning_rate': 1.9200000000000003e-05, 'epoch': 0.12}
{'loss': 0.1458, 'grad_norm': 3.4953455924987793, 'learning_rate': 1.8933333333333334e-05, 'epoch': 0.16}
{'loss': 0.1195, 'grad_norm': 2.3119609355926514, 'learning_rate': 1.866666666666667e-05, 'epoch': 0.2}
{'loss': 0.1489, 'grad_norm': 3.5621891021728516, 'learning_rate': 1.8400000000000003e-05, 'epoch': 0.24}
{'loss': 0.1195, 'grad_norm': 3.5320286750793457, 'learning_rate': 1.8133333333333335e-05, 'epoch': 0.28}
{'loss': 0.1595, 'grad_norm': 4.720970630645752, 'learning_rate': 1.7866666666666666e-05, 'epoch': 0.32}
{'loss': 0.1144, 'grad_norm': 3.1825973987579346, 'learning_rate': 1.76e-05, 'epoch': 0.36}
{'loss': 0.0922, 'grad_norm': 2.528242588043213, 'learning_rate':

  0%|          | 0/63 [00:00<?, ?it/s]

/tmp/ipykernel_21067/2340784742.py:3: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  predictions = [int(p > 0.5) for p in predictions]


{'eval_loss': 0.0760849118232727, 'eval_accuracy': 0.904, 'eval_f1': 0.8753246753246753, 'eval_runtime': 10.0065, 'eval_samples_per_second': 199.871, 'eval_steps_per_second': 6.296, 'epoch': 1.0}
{'loss': 0.0957, 'grad_norm': 2.33520770072937, 'learning_rate': 1.3066666666666668e-05, 'epoch': 1.04}
{'loss': 0.0808, 'grad_norm': 4.529998302459717, 'learning_rate': 1.2800000000000001e-05, 'epoch': 1.08}
{'loss': 0.0669, 'grad_norm': 3.172510862350464, 'learning_rate': 1.2533333333333336e-05, 'epoch': 1.12}
{'loss': 0.0609, 'grad_norm': 1.9965033531188965, 'learning_rate': 1.2266666666666667e-05, 'epoch': 1.16}
{'loss': 0.0637, 'grad_norm': 3.834648370742798, 'learning_rate': 1.2e-05, 'epoch': 1.2}
{'loss': 0.0792, 'grad_norm': 5.726282119750977, 'learning_rate': 1.1733333333333335e-05, 'epoch': 1.24}
{'loss': 0.0818, 'grad_norm': 3.2510621547698975, 'learning_rate': 1.1466666666666668e-05, 'epoch': 1.28}
{'loss': 0.0683, 'grad_norm': 7.002705097198486, 'learning_rate': 1.1200000000000001

...

{'eval_loss': 0.06818950176239014, 'eval_accuracy': 0.909, 'eval_f1': 0.8845177664974619, 'eval_runtime': 11.9741, 'eval_samples_per_second': 167.028, 'eval_steps_per_second': 5.261, 'epoch': 2.0}
{'loss': 0.0535, 'grad_norm': 1.7279194593429565, 'learning_rate': 6.4000000000000006e-06, 'epoch': 2.04}
{'loss': 0.0515, 'grad_norm': 3.3006253242492676, 'learning_rate': 6.133333333333334e-06, 'epoch': 2.08}
{'loss': 0.0576, 'grad_norm': 5.581780910491943, 'learning_rate': 5.8666666666666675e-06, 'epoch': 2.12}
{'loss': 0.0703, 'grad_norm': 3.507948875427246, 'learning_rate': 5.600000000000001e-06, 'epoch': 2.16}
{'loss': 0.0612, 'grad_norm': 6.253634929656982, 'learning_rate': 5.333333333333334e-06, 'epoch': 2.2}
{'loss': 0.0596, 'grad_norm': 2.0923378467559814, 'learning_rate': 5.0666666666666676e-06, 'epoch': 2.24}
{'loss': 0.0669, 'grad_norm': 3.6019699573516846, 'learning_rate': 4.800000000000001e-06, 'epoch': 2.28}
{'loss': 0.0441, 'grad_norm': 2.6677727699279785, 'learning_rate': 4.

  0%|          | 0/63 [00:00<?, ?it/s]

/tmp/ipykernel_21067/2340784742.py:3: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  predictions = [int(p > 0.5) for p in predictions]


{'eval_loss': 0.07142410427331924, 'eval_accuracy': 0.9095, 'eval_f1': 0.8846398980242193, 'eval_runtime': 11.8422, 'eval_samples_per_second': 168.888, 'eval_steps_per_second': 5.32, 'epoch': 3.0}
{'train_runtime': 474.84, 'train_samples_per_second': 50.543, 'train_steps_per_second': 1.579, 'train_loss': 0.08731024392445882, 'epoch': 3.0}


TrainOutput(global_step=750, training_loss=0.08731024392445882, metrics={'train_runtime': 474.84, 'train_samples_per_second': 50.543, 'train_steps_per_second': 1.579, 'total_flos': 1552456398705984.0, 'train_loss': 0.08731024392445882, 'epoch': 3.0})

## Step10 模型评估

In [ ]:
trainer.evaluate(tokenized_datasets["test"])

  0%|          | 0/63 [00:00<?, ?it/s]

/tmp/ipykernel_21067/2340784742.py:3: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  predictions = [int(p > 0.5) for p in predictions]


{'eval_loss': 0.07142410427331924,
 'eval_accuracy': 0.9095,
 'eval_f1': 0.8846398980242193,
 'eval_runtime': 11.8184,
 'eval_samples_per_second': 169.227,
 'eval_steps_per_second': 5.331,
 'epoch': 3.0}

## Step11 模型预测

In [ ]:
from transformers import pipeline, TextClassificationPipeline

In [ ]:
model.config.id2label = {0: "不相似", 1: "相似"}

In [ ]:
pipe = pipeline("text-classification", model=model, tokenizer=tokenizer, device=0)

In [ ]:
# num_labels = 1时的二分类存在bug，所以手动进行标签设定。
result = pipe({"text": "我喜欢北京", "text_pair": "天气怎样"}, function_to_apply="none")
result["label"] = "相似" if result["score"] > 0.5 else "不相似"
result

{'label': '不相似', 'score': 0.19294127821922302}